# 08 · Final model and `submission.csv`

1. **Choose** the final model for each task using the validation set only (never the Kaggle leaderboard).
   Candidates, all built from the validation predictions saved by notebooks 02, 04 and 05:
   * the best k-NN, the best Random Forest
   * an ensemble of 5 tuned networks with different seeds (average of their predicted probabilities)
   * averages of the 5-network ensemble with k-NN, and with k-NN + Random Forest

   We deliberately do **not** use the single seed-42 network's score as a candidate: that run *is* the
   winner of notebook 05's search on this same validation set, so its score is inflated by the selection.
   The fair estimate for one tuned network is the 5-seed mean (printed below for reference), and the
   5-seed ensemble is both at least as accurate and more stable on new data.
2. **Retrain** the chosen model on **all** labelled training rows (train + validation = 20,000 /
   12,000), with the preprocessing re-fitted on those rows. For networks the number of epochs is
   fixed to what validation chose (there is no validation set left to stop on). For k-NN,
   "retraining" just means the test images can now find neighbours among all 20,000 training rows.
3. **Predict** the 10,000 test rows and write `submission.csv`, matched to `sample_submission.csv` by id.

In [1]:
import sys
import json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from src.common import *
from src.knn import KNNClassifier
from src.random_forest import RandomForestClassifier
from src.neural_net import train_mlp, predict_proba
set_seed()
torch.set_num_threads(1)   # small networks: one thread is fastest
N_SEEDS = 5

## 1 · Choose the final model per task (validation accuracy)

In [2]:
CANDIDATES = {                       # name -> members whose probabilities are averaged
    "k-NN": ["knn"],
    "Random Forest": ["rf"],
    "NN ensemble (5 seeds)": ["nn5"],
    "NN ensemble + k-NN": ["nn5", "knn"],
    "NN ensemble + k-NN + RF": ["nn5", "knn", "rf"],
}
choice, cand_rows = {}, []
for task in ("image", "sentiment"):
    y_va = get_split(task)[3]
    val_proba = {"nn1": np.load(PRED_DIR / f"{task}_nn.npz")["proba"],
                 "nn5": np.load(PRED_DIR / f"{task}_nn_ensemble5_proba.npy"),
                 "knn": np.load(PRED_DIR / f"{task}_knn.npz")["proba"],
                 "rf": np.load(PRED_DIR / f"{task}_rf.npz")["proba"]}
    accs = {}
    for name, members in CANDIDATES.items():
        p = np.mean([val_proba[m] for m in members], axis=0)
        accs[name] = accuracy(y_va, p.argmax(1))
        cand_rows.append(dict(task=task, candidate=name, val_acc=accs[name]))
    choice[task] = max(accs, key=accs.get)
    nn_info = json.load(open(PRED_DIR / f"{task}_nn.json"))
    print(f"{task}: single tuned network, mean over 5 seeds = {nn_info['seeds_mean_acc']:.4f} "
          f"(seed 42 alone, the search winner: {nn_info['val_acc']:.4f})")
cand = pd.DataFrame(cand_rows)
save_table(cand, "final_candidates")
print("chosen:", choice)
cand.pivot(index="candidate", columns="task", values="val_acc").loc[list(CANDIDATES)].round(4)

image: single tuned network, mean over 5 seeds = 0.3881 (seed 42 alone, the search winner: 0.3915)
sentiment: single tuned network, mean over 5 seeds = 0.6795 (seed 42 alone, the search winner: 0.6829)
chosen: {'image': 'k-NN', 'sentiment': 'NN ensemble (5 seeds)'}


task,image,sentiment
candidate,,
k-NN,0.5805,0.5933
Random Forest,0.3662,0.6029
NN ensemble (5 seeds),0.4095,0.6800
NN ensemble + k-NN,0.5100,0.6692
NN ensemble + k-NN + RF,0.5135,0.6637


## 2 · Retrain on all training data and predict the test set

In [3]:
def fit_predict_member(member, task, X, y, X_test):
    """Train one member on ALL training rows and return its test-set probabilities."""
    if member in ("nn1", "nn5"):
        info = json.load(open(PRED_DIR / f"{task}_nn.json"))
        cfg = dict(info["config"]); cfg["hidden"] = tuple(cfg["hidden"])
        cfg["epochs"] = int(np.median(info["best_epochs_5_seeds"]))   # epochs chosen on validation
        pp = Preprocessor(info["preprocessing"]).fit(X)               # re-fitted on ALL training rows
        A, T = pp.transform(X), pp.transform(X_test)
        probs = []
        for s in range(1 if member == "nn1" else N_SEEDS):
            model, _, _ = train_mlp(A, y, n_classes=N_CLASSES[task], seed=SEED + s, **cfg)
            probs.append(predict_proba(model, T))
        return np.mean(probs, axis=0), f"{len(probs)} x MLP, preprocessing={info['preprocessing']}, {cfg}"
    if member == "knn":
        cfg = json.load(open(PRED_DIR / f"{task}_knn.json"))["config"]
        pp = Preprocessor(cfg["preprocessing"]).fit(X)
        knn = KNNClassifier(k=cfg["k"], metric=cfg["metric"], weights=cfg["weights"], power=cfg["power"],
                            n_classes=N_CLASSES[task]).fit(pp.transform(X), y)
        return knn.predict_proba(pp.transform(X_test)), f"k-NN {cfg}"
    if member == "rf":
        cfg = json.load(open(PRED_DIR / f"{task}_rf.json"))["config"]
        rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, **cfg).fit(X, y, N_CLASSES[task])
        return rf.predict_proba(X_test), f"Random Forest {cfg}"

test_pred, final_desc = {}, {}
for task in ("image", "sentiment"):
    X, y, X_test, test_ids = load_task(task)
    probs, parts = [], []
    with Timer() as t:
        for member in CANDIDATES[choice[task]]:
            p, desc = fit_predict_member(member, task, X, y, X_test)
            probs.append(p); parts.append(desc)
    p_final = np.mean(probs, axis=0)
    print(f"{task}: final model = {choice[task]}  (trained on {len(y)} rows in {t.seconds:.0f}s)")
    test_pred[task] = pd.DataFrame({"id": test_ids, "label": p_final.argmax(1)})
    final_desc[task] = dict(choice=choice[task], members=parts,
                            validation_accuracy=float(cand[(cand.task == task) & (cand.candidate == choice[task])].val_acc.iloc[0]))
save_json(final_desc, "final_model")
final_desc

image: final model = k-NN  (trained on 20000 rows in 2s)


sentiment: final model = NN ensemble (5 seeds)  (trained on 12000 rows in 53s)


{'image': {'choice': 'k-NN',
  'members': ["k-NN {'preprocessing': 'hellinger+standardize', 'metric': 'cosine', 'k': 100, 'weights': 'distance', 'power': 16.0}"],
  'validation_accuracy': 0.5805},
 'sentiment': {'choice': 'NN ensemble (5 seeds)',
  'members': ["5 x MLP, preprocessing=none, {'hidden': (512, 256), 'activation': 'relu', 'dropout': 0.5, 'lr': 0.001, 'optimizer': 'adam', 'weight_decay': 0.0, 'batch_size': 128, 'epochs': 9}"],
  'validation_accuracy': 0.68}}

## 3 · Build and check `submission.csv`
Exactly the procedure from the project description: merge on `id` with the template (the
template's row order differs from the test files), then validate.

In [4]:
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
predictions = pd.concat([test_pred["image"], test_pred["sentiment"]], ignore_index=True)
submission = sample[["id"]].merge(predictions, on="id", how="left", validate="one_to_one")

assert len(submission) == 10000
assert submission["id"].is_unique
assert submission["label"].notna().all()
submission["label"] = submission["label"].astype(int)
image_mask = submission["id"].str.startswith("img_")
sentiment_mask = submission["id"].str.startswith("text_")
assert image_mask.sum() == 5000 and sentiment_mask.sum() == 5000
assert submission.loc[image_mask, "label"].between(0, 49).all()
assert submission.loc[sentiment_mask, "label"].between(0, 2).all()
assert list(submission.columns) == ["id", "label"]

submission.to_csv(ROOT / "submission.csv", index=False)
print("saved", ROOT / "submission.csv")
submission.head()

saved /home/claude/COMP472-Project/submission.csv


,id,label
0,text_003234,2
1,text_004517,1
2,img_000140,39
3,img_004880,15
4,text_002494,0


### Sanity check: distribution of predicted labels on the test set
The image test set has exactly 100 images per class, so predictions should be roughly flat.
The sentiment test set's class mix is not given.

In [5]:
img_counts = submission.loc[image_mask, "label"].value_counts()
print("image: predicted images per class: min %d, median %d, max %d" % (img_counts.min(), img_counts.median(), img_counts.max()))
sent_counts = submission.loc[sentiment_mask, "label"].value_counts().sort_index()
print("sentiment predictions:", {SENTIMENT_NAMES[k]: int(v) for k, v in sent_counts.items()})

image: predicted images per class: min 44, median 96, max 175
sentiment predictions: {'Negative': 2289, 'Neutral': 1444, 'Positive': 1267}


* **Images:** the test set has exactly 100 images per class, but k-NN predicts some classes far more
  often than others (it did the same on validation). Some classes are "hubs" that many images sit close to.
* **Sentiment:** 46% of test tweets are predicted Negative (38% on the balanced validation set), well above the
  one-third a balanced set would give. Either the test set contains more negative tweets than the balanced training set
  (TweetEval's original test split is not balanced), or test tweets differ in style. We cannot check
  without labels, and we did not adjust anything based on the test set.